In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import DPOTrainer

from peft import LoraConfig, get_peft_model

/workspace/repos/dragon-llm/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Data

## Loading

In [2]:
cache_dir = "/workspace/hf_cache"
dataset_name = "ewin-reg/Stock-Market-Trading-Signals"  # "argilla/llama-2-banking-preference"
banking_pref_ds = load_dataset(dataset_name, cache_dir=cache_dir)

## Formatting

In [3]:
def hard_negative_verdict(correct_verdict: str) -> str:
    "See suggestion here: https://huggingface.co/datasets/ewin-reg/Stock-Market-Trading-Signals"
    match correct_verdict:
        case "BUY":
            return "HOLD"
        case "SELL":
            return "HOLD"
        case "HOLD":
            return "BUY"
        case _:
            raise ValueError(f"Unknown verdict: {correct_verdict}")

def format_prompt(system_prompt: dict[str, str], user_prompt: dict[str, str]) -> str:
    return {
        "role": user_prompt["role"],
        "content": f"{system_prompt['content']}\n{user_prompt['content']}",
    }

def extract_verdict(answer: str) -> str:
    return answer.split("VERDICT: ")[-1]

def format_data_dpo(sample: dict[str, list[dict[str, str]]]) -> dict[str, str]:
    messages = sample["messages"]

    # Create hard negative
    correct_answer = messages[2]
    correct_verdict = extract_verdict(correct_answer['content'])
    wrong_verdict = hard_negative_verdict(correct_verdict)
    wrong_answer = correct_answer.copy()
    wrong_answer['content'] = wrong_answer['content'].replace(f"VERDICT: {correct_verdict}", f"VERDICT: {wrong_verdict}")

    return {
        "prompt": messages[:2], # format_prompt(system_prompt, user_prompt),
        "chosen": [correct_answer],
        "rejected": [wrong_answer],
        # "correct_verdict": correct_verdict,
        # "wrong_verdict": wrong_verdict,
    }


In [4]:
dpo_ds = banking_pref_ds.map(format_data_dpo, remove_columns=["messages"])
dpo_ds["train"][0]

{'prompt': [{'role': 'system',
   'content': 'You are a financial analyst specializing in stock direction prediction. You analyze technical indicators and historical prices to predict stock direction, generating step-by-step reasoning chains followed by a final VERDICT: BUY, SELL, or HOLD.'},
  {'role': 'user',
   'content': 'Analyze AAPL as of 2018-08-31 and predict the stock direction.\n\nPrice history (last 20 days): $49.01 -> $48.80 -> $49.22 -> $49.39 -> $49.71 -> $51.45 -> $50.84 -> $50.95 -> $51.53 -> $52.72 -> $53.82 -> $53.82 (current)\nCurrent price: $53.82\n\nTechnical indicators:\n- RSI(14): 89.3\n- Volume Multiplier: 1.62x\n- Volatility Multiplier: 0.93x\n\nBased on this data, should an investor BUY, SELL, or HOLD AAPL? Answer with a step-by-step reasoning chain and a final VERDICT.'}],
 'chosen': [{'role': 'assistant',
   'content': '<think>\n1. Momentum: RSI is 89.3, indicating an overbought (> 68) state.\n2. Liquidity: Volume Multiplier is 1.62x, showing elevated volume

# Model
## Loading

In [5]:
model_name = "OVHaiLLM/Llama-Open-Finance-8B"
cache_dir = "/workspace/hf_cache"
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    cache_dir=cache_dir
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    cache_dir=cache_dir,
    device_map="auto",  # cpu
    dtype="auto",
)
model

Loading weights: 100%|██████████| 291/291 [00:23<00:00, 12.48it/s]


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096, padding_idx=128009)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4

## PEFT with LoRA

In [6]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

peft_model = get_peft_model(model, lora_config)
peft_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096, padding_idx=128009)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_p

In [7]:
print("Total number of parameters: ", round(peft_model.get_model_status().total_params / 1e9, 2), "G")
print("Number of trainable parameters: ", round(peft_model.get_model_status().trainable_params / 1e6, 2), "M")

Total number of parameters:  8.03 G
Number of trainable parameters:  3.41 M


# DPO training

In [8]:
tokenizer.pad_token_id

128009

In [9]:
dpo_trainer = DPOTrainer(
    model=peft_model,
    train_dataset=dpo_ds["train"],
    eval_dataset=dpo_ds["validation"],
    processing_class=tokenizer,
)

In [10]:
dpo_trainer.train()

Step,Training Loss
10,0.692961


KeyboardInterrupt: 